In [ ]:
import pandas as pd
from rdkit.Chem import PandasTools
from rdkit.Chem import rdFMCS
from rdkit.Chem import Draw
from rdkit import Chem
from rdkit import Chem
from rdkit import RDLogger
from rdkit.Chem import SaltRemover
from rdkit.Chem.MolStandardize import rdMolStandardize
import scipy.stats as st 

from rdkit import DataStructs
from rdkit.Chem import rdFingerprintGenerator

from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import roc_auc_score, r2_score, mean_squared_error
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.ensemble import RandomForestClassifier

import seaborn as sns
import numpy as np
from rdkit.Chem import rdmolops
import matplotlib.pyplot as plt
import pickle as pkl

In [ ]:
project_settings = {"Renin": {"ecfp_diameter": 4, "maximum_depth": 20, "regressor": True},
                    "ALDH1A1": {"ecfp_diameter": 6, "maximum_depth": 20, "regressor": True},
                    "PDE10A": {"ecfp_diameter": 4, "maximum_depth": 20, "regressor": True},
                    "FXA": {"ecfp_diameter": 4, "maximum_depth": 20, "regressor": True},
                    "MMP38": {"ecfp_diameter": 6, "maximum_depth": 20, "regressor": True},
                    "PPARd": {"ecfp_diameter": 6, "maximum_depth": 20, "regressor": True},
                    "Solubility": {"ecfp_diameter": 6, "maximum_depth": 20, "regressor": False},
                    "Lipophilicity": {"ecfp_diameter": 6, "maximum_depth": 20, "regressor": True},
                    "Permeability": {"ecfp_diameter": 4, "maximum_depth": None, "regressor": True}
                    }

## Define variables

In [ ]:
project = "Renin" # Select which project to work on

datapath="" # file path for raw potency data
smiles_col = "Smiles"
y_value = "Potency"

store_path = "" # file path to store trained objective model - assumes pickle file

## Run model training

In [ ]:
df = pd.read_csv(datapath)

In [ ]:
PandasTools.AddMoleculeColumnToFrame(df, smilesCol = smiles_col, molCol="Mol")

Failed to patch pandas - unable to change molecule rendering


In [8]:
df["SMILES_STANDARD"] = df["Smiles"].apply(filter_and_normalize_smiles)

[15:08:19] Running Uncharger
[15:08:19] Running Uncharger
[15:08:19] Running Uncharger
[15:08:19] Running Uncharger
[15:08:19] Running Uncharger
[15:08:19] Running Uncharger
[15:08:19] Running Uncharger
[15:08:19] Running Uncharger
[15:08:19] Running Uncharger
[15:08:19] Running Uncharger
[15:08:19] Running Uncharger
[15:08:19] Running Uncharger
[15:08:19] Running Uncharger
[15:08:19] Running Uncharger
[15:08:19] Running Uncharger
[15:08:19] Running Uncharger
[15:08:19] Running Uncharger
[15:08:19] Running Uncharger
[15:08:19] Running Uncharger
[15:08:19] Running Uncharger
[15:08:19] Running Uncharger
[15:08:19] Running Uncharger
[15:08:19] Running Uncharger
[15:08:19] Running Uncharger
[15:08:19] Running Uncharger
[15:08:19] Running Uncharger
[15:08:19] Running Uncharger
[15:08:19] Running Uncharger
[15:08:19] Running Uncharger
[15:08:19] Running Uncharger
[15:08:19] Running Uncharger
[15:08:19] Running Uncharger
[15:08:19] Running Uncharger
[15:08:19] Running Uncharger
[15:08:19] Run

In [ ]:
df = df[df[smiles_col].notna()]

In [12]:
len(df)

441

In [13]:
df = df.reset_index(drop = True)

In [ ]:
mfpgen = rdFingerprintGenerator.GetMorganGenerator(fpSize=2048, radius=project_settings[project]["ecfp_diameter"]/2)

X = np.array([mfpgen.GetFingerprint(Chem.MolFromSmiles(smi)) for smi in df[smiles_col]])
y = df[y_value].values

In [ ]:
from sklearn.metrics import average_precision_score


kf = KFold(n_splits=5, random_state=42, shuffle = True)
p = []
rmse = []
y_true = []
y_pred = []
results =[]
for i, (train_index, test_index) in enumerate(kf.split(X)):
    if project_settings[project]["regressor"]:
        clf = RandomForestRegressor(max_depth=project_settings[project]["maximum_depth"], n_estimators=100)
        clf.fit(X[train_index], y[train_index])
        y_score = clf.predict(X[test_index])
        p.append(st.spearmanr(y[test_index], y_score).statistic)
        rmse.append(np.sqrt(mean_squared_error(y[test_index], y_score)))
        print("Fold", i, r2_score(y[test_index], y_score), np.sqrt(mean_squared_error(y[test_index], y_score)), st.spearmanr(y[test_index], y_score))
        results.append([i, r2_score(y[test_index], y_score), st.spearmanr(y[test_index], y_score).statistic])

        y_true.extend(y[test_index])
        y_pred.extend(y_score)
    else:
        clf = RandomForestClassifier(max_depth=20, n_estimators=100)
        clf.fit(X[train_index], y[train_index])
        y_score = clf.predict_proba(X[test_index])
        print("Fold", i, roc_auc_score(y[test_index], y_score[:, 1]), average_precision_score(y[test_index], y_score[:, 1]))
        results.append([i, roc_auc_score(y[test_index], y_score[:, 1]), average_precision_score(y[test_index], y_score[:, 1])])



Fold 0 0.7558397499359877 0.8915847949457671 SignificanceResult(statistic=0.80627887771155, pvalue=1.5282545147526123e-21)
Fold 1 0.8228496293659765 0.757307342276026 SignificanceResult(statistic=0.8687502748602515, pvalue=5.596025106980549e-28)
Fold 2 0.6422255846300724 0.9620537560963821 SignificanceResult(statistic=0.7816373580221248, pvalue=2.5540711893272435e-19)
Fold 3 0.7430234928901931 0.8491051260048084 SignificanceResult(statistic=0.791236425246827, pvalue=4.60203492034314e-20)
Fold 4 0.6648524303931331 0.9875350935970048 SignificanceResult(statistic=0.7850820164001344, pvalue=1.394834737937245e-19)


In [ ]:
if project_settings[project]["regressor"]:
    results = pd.DataFrame(results, columns = ["Fold", "R2", "Rank"])
else:
    results = pd.DataFrame(results, columns = ["Fold", "AUROC", "AUPR"])


In [18]:
results.mean()

Fold    2.000000
R2      0.725758
Rank    0.806597
dtype: float64

In [20]:
clf.fit(X, y)

RandomForestRegressor(max_depth=20)

In [ ]:
with open(store_path, "wb") as f:
    pkl.dump(clf, f, protocol=5)